### Environment Example

In [1]:
import numpy as np
from env.pusht.pusht_env import PushTEnv   # run from the dino_wm repo root

env = PushTEnv(
    render_size=224,     # DINO patch grid
    with_target=False,   # hide the green goal overlay
    relative=True,       # delta actions
    action_scale=100,
    shape="T",
)

obs, state = env.reset()          # note: 2-tuple, not (obs, info)
# obs = {"visual": (224,224,3) uint8, "proprio": (2,) agent xy}
# state = [agent_x, agent_y, block_x, block_y, block_angle]

for t in range(100):
    action = np.random.randn(2) * 0.3          # delta, pre-scale
    obs, reward, done, info = env.step(action) # 4-tuple, old gym API
    env.render(mode='human')
    # done is ALWAYS False here

env.close()
print(info["max_coverage"], info["final_coverage"])

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


0.0 0.0


### Dataset Example

In [2]:
from dataset.pusht_dset import load_pusht_slice_train_val
from torchvision import transforms
transform = transforms.Compose([transforms.Resize((224, 224))])
dset, traj_dset = load_pusht_slice_train_val(
    transform=None,    
    n_rollout=None,
    data_path='dataset/data/pusht_noise',
    normalize_action=True,
    split_ratio=0.8,
    num_hist=3,
    num_pred=1,
    frameskip=5,
    with_velocity=True,
)

Loaded 18685 rollouts
Loaded 21 rollouts


In [3]:
train_dset = dset['train']
valid_dset = dset['valid']
len(train_dset), len(valid_dset)

(1981721, 2115)

In [4]:
sample = train_dset[0]
obs = sample[0]['visual']
action = sample[1]
obs.shape, action.shape
# actions are concatenated across frameskips
# dset are slices of length num_hist+num_pred

(torch.Size([4, 3, 224, 224]), torch.Size([4, 10]))

In [5]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dset, batch_size=16, shuffle=True)
valid_loader = DataLoader(valid_dset, batch_size=16)

for batch in train_loader:
    obs = batch[0]['visual']
    action = batch[1]
    print(obs.shape, action.shape)
    break

torch.Size([16, 4, 3, 224, 224]) torch.Size([16, 4, 10])


### Model Example

In [6]:
from model.lewm import LeWorldModel
import torch

model = LeWorldModel()
optimizer = torch.optim.AdamW(params=model.parameters(), lr=1e-4)
for batch in train_loader:
    obs = batch[0]['visual']
    action = batch[1]
    loss_dict = model(obs, action)
    optimizer.zero_grad()
    loss_dict['loss'].backward()
    optimizer.step()
    break

loss_dict

{'loss': tensor(2.0057, grad_fn=<AddBackward0>),
 'pred_loss': tensor(1.9677, grad_fn=<MeanBackward0>),
 'sigreg_loss': tensor(0.3800, grad_fn=<MulBackward0>)}

### Training Example

In [ ]:
from train import train

train_losses, valid_losses, batch_losses = train(
    epochs=1,
    train_loader=valid_loader, # for a shorter loop
    valid_loader=valid_loader, 
    model=model, 
    optimizer=optimizer,
    resume_path=None, 
    log_batch_interval=10,
    save_dir='./runs', 
    save_name='example', 
    save_epoch_interval=1,
    device='cuda'
)

Epoch 1 Training...

    Batch 0, loss: 2.02116, pred_loss: 1.98378, sigreg_loss: 0.37390
    Batch 10, loss: 2.05739, pred_loss: 2.01953, sigreg_loss: 0.37861
    Batch 20, loss: 2.03773, pred_loss: 2.00117, sigreg_loss: 0.36561
    Batch 30, loss: 1.99461, pred_loss: 1.95774, sigreg_loss: 0.36868
    Batch 40, loss: 2.05737, pred_loss: 2.02219, sigreg_loss: 0.35179
    Batch 50, loss: 2.02528, pred_loss: 1.99123, sigreg_loss: 0.34043
    Batch 60, loss: 2.03437, pred_loss: 1.99944, sigreg_loss: 0.34933
    Batch 70, loss: 1.99674, pred_loss: 1.96296, sigreg_loss: 0.33782
    Batch 80, loss: 2.00407, pred_loss: 1.96947, sigreg_loss: 0.34597
    Batch 90, loss: 1.97369, pred_loss: 1.93977, sigreg_loss: 0.33920
    Batch 100, loss: 2.00758, pred_loss: 1.97360, sigreg_loss: 0.33979
    Batch 110, loss: 1.98648, pred_loss: 1.95254, sigreg_loss: 0.33943
    Batch 120, loss: 1.97202, pred_loss: 1.93906, sigreg_loss: 0.32965
    Batch 130, loss: 1.96382, pred_loss: 1.92882, sigreg_loss: 0.35

### Planning example

In [1]:
import torch
from einops import rearrange
from torchvision import transforms
from dataset.pusht_dset import load_pusht_slice_train_val, ACTION_MEAN, ACTION_STD
from env.pusht.pusht_env import PushTEnv
from model.lewm import LeWorldModel
from planning import model_predictive_control

hist_len = 3
frameskip = 5
goal_offset = 5          
episode = 0
transform = transforms.Resize((224, 224))

_, traj_dsets = load_pusht_slice_train_val(
    transform=transform, 
    n_rollout=None,
    data_path='dataset/data/pusht_noise',
    num_hist=hist_len, 
    num_pred=1, 
    frameskip=frameskip,
)
val_dset = traj_dsets["valid"]

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


Loaded 18685 rollouts
Loaded 21 rollouts


In [2]:
context_frames = [i * frameskip for i in range(hist_len)]        # [0, 5, 10]
goal_frame = context_frames[-1] + goal_offset * frameskip        # 35

In [3]:
obs, _, state, _ = val_dset.get_frames(episode, context_frames + [goal_frame])
o_init = obs["visual"][:hist_len]        # [3, 3, 224, 224]
o_goal = obs["visual"][-1]               # [3, 224, 224]

raw_actions = val_dset.actions[episode, :(hist_len - 1) * frameskip]   # [10, 2]
a_init = rearrange(raw_actions, "(n f) d -> n (f d)", n=hist_len - 1)  # [2, 10]

In [ ]:
env = PushTEnv(render_size=224, with_target=False, with_velocity=True)
env.seed(0)
env.reset_to_state = state[hist_len - 1].numpy()
env.reset()

model = LeWorldModel().eval()
#model.load_state_dict(torch.load("checkpoints/lewm.pt", map_location="cpu"))

coverage, costs = model_predictive_control(
    model, env, o_goal, o_init, a_init,
    action_mean=ACTION_MEAN, action_std=ACTION_STD, transform=transform,
)

In [6]:
coverage, costs

(0.0, [483.88079833984375, 483.9228515625, 483.91943359375])